In [1]:
import os
import pandas as pd

files = sorted(os.listdir("data/raw"))
df = pd.concat([pd.read_csv("data/raw/" + f) for f in files], ignore_index=True)
print(df.shape)
df["Period"].value_counts()

(367919, 121)


Period
RTT-March-2026    184184
RTT-April-2025    183735
Name: count, dtype: int64

In [2]:
df = df[df["RTT Part Type"] == "Part_2"].copy()
df.shape

(129471, 121)

In [3]:
week_cols = [c for c in df.columns if c.startswith("Gt ")]
num_cols = week_cols + ["Total", "Patients with unknown clock start date", "Total All"]
df[num_cols] = df[num_cols].fillna(0)

print("Number of week columns:", len(week_cols))
print("First:", week_cols[0], "| 18th:", week_cols[17], "| 53rd:", week_cols[52], "| Last:", week_cols[-1])

Number of week columns: 105
First: Gt 00 To 01 Weeks SUM 1 | 18th: Gt 17 To 18 Weeks SUM 1 | 53rd: Gt 52 To 53 Weeks SUM 1 | Last: Gt 104 Weeks SUM 1


In [4]:
is_total = df["Treatment Function Name"] == "Total"
print("NHS 'Total' rows:     ", df.loc[is_total, "Total All"].sum())
print("Sum of specialties:   ", df.loc[~is_total, "Total All"].sum())
df = df[~is_total].copy()

NHS 'Total' rows:      14466963
Sum of specialties:    14466963


In [5]:
df["within_18wks"] = df[week_cols[:18]].sum(axis=1)
df["over_52wks"] = df[week_cols[52:]].sum(axis=1)
df["month"] = pd.to_datetime(df["Period"].str.replace("RTT-", ""), format="%B-%Y")
df[["Provider Org Name", "Treatment Function Name", "Total All", "Total", "within_18wks", "over_52wks"]].head()

,Provider Org Name,Treatment Function Name,Total All,Total,within_18wks,over_52wks
0,BUCKSHAW HOSPITAL,General Surgery Service,1,0.0,1.0,0.0
1,BUCKSHAW HOSPITAL,Ear Nose and Throat Service,1,0.0,0.0,0.0
3,BUCKSHAW HOSPITAL,General Surgery Service,1,0.0,1.0,0.0
7,BUCKSHAW HOSPITAL,General Surgery Service,1,0.0,1.0,0.0
8,BUCKSHAW HOSPITAL,Urology Service,1,0.0,1.0,0.0


In [6]:
df["known_start"] = df[week_cols].sum(axis=1)

print("Rows where NHS 'Total' disagrees with week columns:", (df["Total"] != df["known_start"]).sum())
gap = (df["known_start"] + df["Patients with unknown clock start date"] - df["Total All"]).abs().sum()
print("Check (should be 0):", gap)

Rows where NHS 'Total' disagrees with week columns: 103669
Check (should be 0): 0.0


In [7]:
df["provider_type"] = df["Provider Org Code"].str[0].isin(["R", "T"]).map({True: "NHS trust", False: "Independent sector"})

clean = df.groupby(["month", "Provider Parent Name", "Provider Org Code", "Provider Org Name", "provider_type", "Treatment Function Name"],
                   as_index=False)[["Total All", "known_start", "within_18wks", "over_52wks"]].sum()
clean = clean.rename(columns={"Provider Parent Name": "icb", "Provider Org Code": "org_code",
                              "Provider Org Name": "trust", "Treatment Function Name": "specialty",
                              "Total All": "waiting_list"})
os.makedirs("data/clean", exist_ok=True)
clean.to_csv("data/clean/rtt_clean.csv", index=False)
clean.shape

(7251, 10)